### Step 1: Imports and Basing Preprocessing:

In [6]:
import pandas as pd
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import optuna
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import mlflow
import mlflow.sklearn
import dagshub
import json
import pickle
from catboost import CatBoostRegressor

In [32]:
mlflow.set_tracking_uri("https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow")

dagshub.init(repo_owner='PriyanshuMewal', repo_name='delivery-time-prediction', mlflow=True)

mlflow.set_experiment("Experiment 4: Model Selection Without Missing Values")

Initialized MLflow to track repo "PriyanshuMewal/delivery-time-prediction"

Repository PriyanshuMewal/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/0159417f09e1476798193b9381e1c2c4', creation_time=1771566150176, experiment_id='4', last_update_time=1771566150176, lifecycle_stage='active', name='Experiment 4: Model Selection Without Missing Values', tags={'mlflow.experimentKind': 'custom_model_development'}>

#### After Imputation:

In [33]:
train_df = pd.read_csv("/content/without_missing_values/train_df.csv")
test_df = pd.read_csv("/content/without_missing_values/test_df.csv")

In [14]:
train_df.head()

,age,ratings,pickup_time,distance,traffic,distance_type,order_time_of_day,weather_fog,weather_missing,weather_sandstorms,...,vehicle_type_motorcycle,vehicle_type_scooter,festival_yes,city_type_semi-urban,city_type_urban,multi_deliveries,condition,order_month,is_weekend,time
0,0.684211,0.84,0.846477,0.622989,1.0,2.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0,3,0,1.671031
1,0.105263,0.80,0.564597,0.003331,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,2,3,0,-0.673246
2,0.894737,0.96,0.934564,0.002487,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,2,2,0,-0.673246
3,0.000000,0.84,0.921980,0.317057,3.0,1.0,2.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,2,3,0,-0.033898
4,0.315789,1.00,0.093960,0.318839,1.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0,3,0,-0.033898


In [15]:
print(train_df.duplicated().sum())
test_df.duplicated().sum()

0


np.int64(0)

In [36]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29532 entries, 0 to 29531
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Unnamed: 0                        29532 non-null  int64  
 1   ord_cat__traffic                  29532 non-null  float64
 2   ord_cat__distance_type            29532 non-null  float64
 3   ord_cat__order_time_of_day        29532 non-null  float64
 4   nom_cat__weather_fog              29532 non-null  float64
 5   nom_cat__weather_sandstorms       29532 non-null  float64
 6   nom_cat__weather_stormy           29532 non-null  float64
 7   nom_cat__weather_sunny            29532 non-null  float64
 8   nom_cat__weather_windy            29532 non-null  float64
 9   nom_cat__order_type_drinks        29532 non-null  float64
 10  nom_cat__order_type_meal          29532 non-null  float64
 11  nom_cat__order_type_snack         29532 non-null  float64
 12  nom_

In [37]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7383 entries, 0 to 7382
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Unnamed: 0                        7383 non-null   int64  
 1   ord_cat__traffic                  7383 non-null   float64
 2   ord_cat__distance_type            7383 non-null   float64
 3   ord_cat__order_time_of_day        7383 non-null   float64
 4   nom_cat__weather_fog              7383 non-null   float64
 5   nom_cat__weather_sandstorms       7383 non-null   float64
 6   nom_cat__weather_stormy           7383 non-null   float64
 7   nom_cat__weather_sunny            7383 non-null   float64
 8   nom_cat__weather_windy            7383 non-null   float64
 9   nom_cat__order_type_drinks        7383 non-null   float64
 10  nom_cat__order_type_meal          7383 non-null   float64
 11  nom_cat__order_type_snack         7383 non-null   float64
 12  nom_ca

In [38]:
train_df.drop(columns=["Unnamed: 0"], inplace=True)
test_df.drop(columns=["Unnamed: 0"], inplace=True)

In [39]:
X_train = train_df.drop(columns=["time"])
X_test = test_df.drop(columns=["time"])

y_train = train_df["time"]
y_test = test_df["time"]

In [40]:
X_train.head()

,ord_cat__traffic,ord_cat__distance_type,ord_cat__order_time_of_day,nom_cat__weather_fog,nom_cat__weather_sandstorms,nom_cat__weather_stormy,nom_cat__weather_sunny,nom_cat__weather_windy,nom_cat__order_type_drinks,nom_cat__order_type_meal,...,nom_cat__city_type_semi-urban,nom_cat__city_type_urban,num__age,num__ratings,num__pickup_time,num__distance,remainder__condition,remainder__multi_deliveries,remainder__order_month,remainder__is_weekend
0,0.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.473684,0.84,0.714765,0.404182,1,0.0,4,1
1,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.631579,0.88,0.740772,0.243678,1,0.0,3,1
2,3.0,3.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.000000,0.64,0.573826,0.787364,0,0.0,4,0
3,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.263158,0.88,0.660235,0.236691,0,0.0,4,1
4,1.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.052632,0.92,0.360738,0.397584,0,1.0,3,1


In [42]:
y_test

,time
0,-0.929363
1,1.633204
2,0.245147
3,-0.288721
4,-1.036137
...,...
7378,0.245147
7379,-1.142910
7380,0.245147
7381,2.060299


#### After removing missing values:

### Experimentation: Hyperparameter Tunning Over XGBoost, LightGBM, Catboost, Random Forest and SVM:

In [49]:
def objactive(trial, model):

    model_name = model().__class__.__name__

    param_dict = {}

    if model_name in [XGBRegressor().__class__.__name__, LGBMRegressor().__class__.__name__]:

        param_dict["n_estimators"] = trial.suggest_int("n_estimators", 50, 300)
        param_dict["max_depth"] = trial.suggest_int("max_depth", 3, 10)
        param_dict["learning_rate"] = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True)

    elif model_name == RandomForestRegressor().__class__.__name__:

        param_dict["n_estimators"] = trial.suggest_int("n_estimators", 50, 300)
        param_dict["max_depth"] = trial.suggest_int("max_depth", 3, 10)
        param_dict["min_samples_split"] = trial.suggest_int("min_samples_split", 2, 20)
        param_dict["min_samples_leaf"] = trial.suggest_int("min_samples_leaf", 1, 20)

    elif model_name == SVR().__class__.__name__:

        param_dict["C"] = trial.suggest_float("C", 0.1, 100, log=True)
        param_dict["epsilon"] = trial.suggest_float("epsilon", 0.005, 0.3)
        param_dict["gamma"] = trial.suggest_float("gamma", 1e-4, 1.0, log=True)

    if model_name == XGBRegressor().__class__.__name__:

        param_dict["tree_method"] = "hist"

    elif model_name == CatBoostRegressor().__class__.__name__:

        param_dict.update({"loss_function": "RMSE",
                          "iterations": trial.suggest_int("iterations", 500, 4000),
                          "learning_rate": trial.suggest_float("learning_rate", 1e-2, 1e-1, log=True),
                          "depth": trial.suggest_int("depth", 5, 9),
                          "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 3, 10),
                          "verbose": 0})


    model = model(**param_dict)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_true=y_test, y_pred=y_pred)
    r2 = r2_score(y_test, y_pred)

    trial.set_user_attr("r2", r2)
    return mae

In [50]:
best_params = {}

# Hyperparameter tunning over XGboost:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, XGBRegressor), n_trials=30)

best_params.update({"Xgboost": study.best_params})

[I 2026-02-20 06:19:50,409] A new study created in memory with name: no-name-d7142ef2-2bca-4cb8-a4c3-30e7bbaaed77
[I 2026-02-20 06:19:55,043] Trial 0 finished with value: 0.6827558548632389 and parameters: {'n_estimators': 99, 'max_depth': 3, 'learning_rate': 0.004240337536858211}. Best is trial 0 with value: 0.6827558548632389.
[I 2026-02-20 06:20:00,024] Trial 1 finished with value: 0.35671534759225265 and parameters: {'n_estimators': 137, 'max_depth': 4, 'learning_rate': 0.06699497687420111}. Best is trial 1 with value: 0.35671534759225265.
[I 2026-02-20 06:20:07,527] Trial 2 finished with value: 0.32533939452665966 and parameters: {'n_estimators': 274, 'max_depth': 9, 'learning_rate': 0.032911689792474746}. Best is trial 2 with value: 0.32533939452665966.
[I 2026-02-20 06:20:08,210] Trial 3 finished with value: 0.46900894087121886 and parameters: {'n_estimators': 224, 'max_depth': 4, 'learning_rate': 0.006912524778613582}. Best is trial 2 with value: 0.32533939452665966.
[I 2026-02

In [51]:
# Hyperparameter tunning over LightGBM:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, LGBMRegressor), n_trials=30)

best_params.update({"lightgbm": study.best_params})

[I 2026-02-20 06:21:09,358] A new study created in memory with name: no-name-50e820d1-3bb7-4810-a487-da3465c096d5


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 29532, number of used features: 24
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

[I 2026-02-20 06:21:10,413] Trial 0 finished with value: 0.39206563741911105 and parameters: {'n_estimators': 251, 'max_depth': 3, 'learning_rate': 0.03025999038955041}. Best is trial 0 with value: 0.39206563741911105.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005211 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 29532, number of used features: 24
[LightGBM] [Info] Start training from score -0.000000


[I 2026-02-20 06:21:10,945] Trial 1 finished with value: 0.7391219373921404 and parameters: {'n_estimators': 84, 'max_depth': 9, 'learning_rate': 0.0014138529909454673}. Best is trial 0 with value: 0.39206563741911105.


...

[I 2026-02-20 06:21:38,158] Trial 27 finished with value: 0.3761598419420892 and parameters: {'n_estimators': 284, 'max_depth': 10, 'learning_rate': 0.007141671019510455}. Best is trial 12 with value: 0.3258124432735564.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003653 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 29532, number of used features: 24
[LightGBM] [Info] Start training from score -0.000000


[I 2026-02-20 06:21:39,143] Trial 28 finished with value: 0.3288447175760593 and parameters: {'n_estimators': 230, 'max_depth': 9, 'learning_rate': 0.04781076992015097}. Best is trial 12 with value: 0.3258124432735564.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003492 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 29532, number of used features: 24
[LightGBM] [Info] Start training from score -0.000000


[I 2026-02-20 06:21:40,238] Trial 29 finished with value: 0.3283656370865855 and parameters: {'n_estimators': 254, 'max_depth': 10, 'learning_rate': 0.036553832450192524}. Best is trial 12 with value: 0.3258124432735564.


In [52]:
# Hyperparameter tunning over RandomForestRegressor:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, RandomForestRegressor), n_trials=30)

best_params.update({"random_forest": study.best_params})

[I 2026-02-20 06:21:44,522] A new study created in memory with name: no-name-84e0054d-5b6a-43fa-aee2-8d7b654d301e
[I 2026-02-20 06:22:09,002] Trial 0 finished with value: 0.3895312746397602 and parameters: {'n_estimators': 267, 'max_depth': 8, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.3895312746397602.
[I 2026-02-20 06:22:25,480] Trial 1 finished with value: 0.48751808088359727 and parameters: {'n_estimators': 287, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.3895312746397602.
[I 2026-02-20 06:22:36,655] Trial 2 finished with value: 0.39253016691558634 and parameters: {'n_estimators': 130, 'max_depth': 8, 'min_samples_split': 14, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.3895312746397602.
[I 2026-02-20 06:22:56,954] Trial 3 finished with value: 0.3656765796431088 and parameters: {'n_estimators': 181, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 15}. Best is trial 3 with v

In [55]:
# Hyperparameter tunning over Catboost:
# study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, CatBoostRegressor), n_trials=10)

best_params.update({"catboost": study.best_params})

[I 2026-02-20 06:39:51,334] Trial 20 finished with value: 0.32407355075680405 and parameters: {'iterations': 3194, 'learning_rate': 0.01292170496928116, 'depth': 9, 'l2_leaf_reg': 5.752526342064876}. Best is trial 20 with value: 0.32407355075680405.
[I 2026-02-20 06:40:52,572] Trial 21 finished with value: 0.32422624955126655 and parameters: {'iterations': 3149, 'learning_rate': 0.012184054916136586, 'depth': 9, 'l2_leaf_reg': 5.694780504601955}. Best is trial 20 with value: 0.32407355075680405.
[I 2026-02-20 06:41:49,798] Trial 22 finished with value: 0.3246953002390611 and parameters: {'iterations': 3235, 'learning_rate': 0.012517332585823571, 'depth': 9, 'l2_leaf_reg': 5.485464159130496}. Best is trial 20 with value: 0.32407355075680405.
[I 2026-02-20 06:42:57,174] Trial 23 finished with value: 0.3242747980980478 and parameters: {'iterations': 2948, 'learning_rate': 0.012402701206708805, 'depth': 9, 'l2_leaf_reg': 4.308545867563107}. Best is trial 20 with value: 0.32407355075680405.

In [ ]:
# Hyperparameter tunning over SVM:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, SVR), n_trials=10)

best_params.update({"svm": study.best_params})

[I 2026-02-19 15:04:07,226] A new study created in memory with name: no-name-9ac3a97d-4b1e-4296-8f5f-3bed162f2431
[I 2026-02-19 15:07:35,212] Trial 0 finished with value: 0.3909910928553759 and parameters: {'C': 1.5999121008456296, 'epsilon': 0.027198370700621213, 'gamma': 0.13864829887964417}. Best is trial 0 with value: 0.3909910928553759.
[I 2026-02-19 15:08:51,835] Trial 1 finished with value: 0.4276625867931324 and parameters: {'C': 0.5737908631589849, 'epsilon': 0.010426425873897563, 'gamma': 0.03160137418109366}. Best is trial 0 with value: 0.3909910928553759.
[I 2026-02-19 15:09:48,111] Trial 2 finished with value: 0.511577126152768 and parameters: {'C': 6.204039879612022, 'epsilon': 0.22102457619176563, 'gamma': 0.0003278634675621763}. Best is trial 0 with value: 0.3909910928553759.
[W 2026-02-19 15:27:29,953] Trial 3 failed with parameters: {'C': 39.19056964604964, 'epsilon': 0.2218436774886583, 'gamma': 0.4379018157596946} because of the following error: KeyboardInterrupt().

KeyboardInterrupt: 

In [ ]:
best_params

In [ ]:
import json

with open("/content/best_params.json", mode="w") as file:
  json.dump(best_params, file, indent=4)

In [ ]:
with open("best_params.json", "r") as file:
    best_params = json.load(file)
best_params

{'Xgboost': {'n_estimators': 52, 'max_depth': 10, 'learning_rate': 0.1},
 'lightgbm': {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.1},
 'random_forest': {'n_estimators': 245,
  'max_depth': 10,
  'min_samples_split': 12,
  'min_samples_leaf': 3},
 'svm': {'C': 0.7016805586928897,
  'epsilon': 0.09532961100132147,
  'gamma': 0.13703551969616495}}

In [56]:
def log_mlflow(model, X_train, y_train, X_test, y_test):

    with mlflow.start_run(run_name=model.__class__.__name__):

        parameters = {"model": model.__class__.__name__,
                      "Imputation": "droped"}

        # logging parameters:
        parameters.update(model.get_params())
        mlflow.log_params(parameters)

        # Train and evaluate model:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # logging metrics:
        mlflow.log_metric("mean_absolute_error", mae)
        mlflow.log_metric("r2_score", r2)

        # log model:
        # signature = mlflow.models.infer_signature(X_test, y_pred)
        # mlflow.sklearn.log_model(model, signature=signature)

        # logging datasets:
        # mlflow.log_artifact("../data/interim/reddit_cleaned.csv")

        # Set tags:
        # mlflow.set_tag("Author", "Priyanshu Mewal")
        # mlflow.set_tag("Description", "Food delivery prediction using Iterative Imputer with max_iter=15.")
        # mlflow.set_tag("Experiment", "HP over iterative imputer")

In [57]:
for model_name, params in best_params.items():

    if model_name == "Xgboost":
       model = XGBRegressor(**params)
    elif model_name == "lightgbm":
        model = LGBMRegressor(**params)
    elif model_name == "random_forest":
        model = RandomForestRegressor(**params)
    elif model_name == "catboost":
        model = CatBoostRegressor(**params)
    else:
        model = SVR(**params)

    log_mlflow(model, X_train, y_train, X_test, y_test)

🏃 View run XGBRegressor at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4/runs/7c90050a5e8b4e549702a3b7cc167d41
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 29532, number of used features: 24
[LightGBM] [Info] Start training from score -0.000000
🏃 View run LGBMRegressor at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4/runs/b905167e8c9f43179fe9ed9c5e02dfa3
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4
🏃 View run RandomForestRegressor at: https://dagshub.com/PriyanshuMewal/delivery-time